In [1]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from slot_util import load_slot_tables, print_df_attrs, get_reel_symbols, get_reelstop_df, add_way_wild_combination_counts, add_scatter_window_counts, get_way_pattern_df, add_way_rtp

In [3]:
reel_df, paytable_df, _, game_settings = load_slot_tables("sheet/Way_R1WithoutWild.xlsx")

In [4]:
print_df_attrs(reel_df)
display(reel_df)

print_df_attrs(paytable_df)
display(paytable_df)

print_df_attrs(game_settings)
display(game_settings)

shape: (20, 5)
表: 滾輪表
列: Stop 索引
行: 第幾個滾輪


,R1,R2,R3,R4,R5
0,K,W,W,W,W
1,Q,K,K,K,K
2,J,Q,Q,Q,Q
3,10,J,J,J,J
4,9,10,10,10,10
5,9,9,9,9,9
6,K,S,S,S,9
7,Q,K,K,K,K
8,J,Q,Q,Q,Q
9,10,J,J,J,J


shape: (6, 3)
表: 賠率表
列: 連線的 Symbol
行: 由左到右的連線數


,3,4,5
Symbol,,,
K,5,30,100
Q,5,25,100
J,5,20,75
10,5,20,75
9,5,10,50
S,50,0,0


shape: (2,)
表: 遊戲設定
列: 設定名稱
值: 設定值


Setting
WindowHeight     3
WayPayBet       50
Name: Value, dtype: int64

In [5]:
symbols = get_reel_symbols(reel_df)
symbols

['W', 'K', 'Q', 'J', '10', '9', 'S']

In [6]:
reelstop_df = get_reelstop_df(reel_df, symbols)

In [7]:
add_way_wild_combination_counts(reelstop_df, reel_df, symbols, game_settings["WindowHeight"])
add_scatter_window_counts(reelstop_df, game_settings["WindowHeight"])

In [8]:
print_df_attrs(reelstop_df)
display(reelstop_df)

shape: (20, 5)
表: Symbol 在各滾輪的 Stop 數
列: Symbol
行: 第幾個滾輪


,R1,R2,R3,R4,R5
Symbol,,,,,
W,0,1,1,1,1
K,2,3,3,3,3
Q,3,3,3,3,3
J,4,5,4,4,4
10,5,3,4,4,4
9,6,4,4,4,5
S,0,1,1,1,0
-,20,20,20,20,20
WK,6,12,12,12,12


In [9]:
way_df = get_way_pattern_df(reel_df, paytable_df, symbols, window_height=game_settings["WindowHeight"])
add_way_rtp(way_df, reelstop_df, paytable_df, game_settings["WindowHeight"], game_settings["WayPayBet"])

In [10]:
print_df_attrs(way_df)
display(
    way_df.style.format({
        "RTP": "{:.2%}",
    })
)

shape: (31, 15)
表: Way 與 Scatter 組合
列: Symbol x MatchCount
行.Ri Factor: 第 i 輪對 Ways 的乘數 (平均有效符號數 / 不接續機率 / 1: 由 pattern 的 K、WK、X_WK、- 決定)
行.Ways: 一般 Symbol：一次 spin 的平均 ways 數；Scatter：該組合的出現機率
行.RTP: Ways × Pay / WayPayBet


,Symbol,MatchCount,R1,R2,R3,R4,R5,R1 Factor,R2 Factor,R3 Factor,R4 Factor,R5 Factor,Ways,Pay,RTP
0,K,5,K,WK,WK,WK,WK,0.300000,0.600000,0.600000,0.600000,0.600000,0.038880,100,7.78%
1,K,4,K,WK,WK,WK,X_WK,0.300000,0.600000,0.600000,0.600000,0.500000,0.032400,30,1.94%
2,K,3,K,WK,WK,X_WK,-,0.300000,0.600000,0.600000,0.500000,1.000000,0.054000,5,0.54%
3,Q,5,Q,WQ,WQ,WQ,WQ,0.450000,0.600000,0.600000,0.600000,0.600000,0.058320,100,11.66%
4,Q,4,Q,WQ,WQ,WQ,X_WQ,0.450000,0.600000,0.600000,0.600000,0.450000,0.043740,25,2.19%
5,Q,3,Q,WQ,WQ,X_WQ,-,0.450000,0.600000,0.600000,0.450000,1.000000,0.072900,5,0.73%
6,J,5,J,WJ,WJ,WJ,WJ,0.600000,0.900000,0.750000,0.750000,0.750000,0.227813,75,34.17%
7,J,4,J,WJ,WJ,WJ,X_WJ,0.600000,0.900000,0.750000,0.750000,0.250000,0.075938,20,3.04%
8,J,3,J,WJ,WJ,X_WJ,-,0.600000,0.900000,0.750000,0.250000,1.000000,0.101250,5,1.01%
9,10,5,10,W10,W10,W10,W10,0.750000,0.600000,0.750000,0.750000,0.750000,0.189844,75,28.48%


In [11]:
print(f'RTP: {way_df["RTP"].sum():.2%}')

RTP: 134.15%
